# Week 1, Notebook 2: Backpropagation from Scratch
## Building the Engine of Deep Learning

**What you'll build:** A complete backpropagation engine for multi-layer networks.

**Curriculum points:**
- ⑤ Backpropagation is just the chain rule (dynamic programming for gradients)
- ② ReLU for better gradient flow
- ④ Neural networks minimize loss, not "understand"

**Time estimate:** 45–60 minutes

---
### The Big Idea
Backprop isn't magic. It's the chain rule applied systematically from output back to input.  
Think of it as: "How much did each weight contribute to the error?"

```
Forward:  input → layer1 → layer2 → output → loss
Backward: input ← layer1 ← layer2 ← output ← loss
                 gradients flow backward
```

## Part 1: The Chain Rule — Visualized

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

# ============================================================
# Chain rule refresher with a concrete example
# ============================================================
# Suppose: L = (relu(w*x + b) - y)²
# 
# Forward: z = w*x + b → a = relu(z) → L = (a - y)²
# 
# Backward (chain rule):
#   dL/da = 2(a - y)
#   da/dz = 1 if z > 0, else 0   (ReLU derivative)  
#   dz/dw = x
#   dz/db = 1
#
#   dL/dw = dL/da · da/dz · dz/dw = 2(a-y) · relu'(z) · x
#   dL/db = dL/da · da/dz · dz/db = 2(a-y) · relu'(z) · 1

# Let's verify with numerical gradients
def numerical_gradient(f, param, idx, eps=1e-5):
    """Compute gradient numerically (for verification)."""
    original = param[idx].copy() if hasattr(param[idx], 'copy') else param[idx]
    
    param[idx] = original + eps
    loss_plus = f()
    
    param[idx] = original - eps
    loss_minus = f()
    
    param[idx] = original  # restore
    return (loss_plus - loss_minus) / (2 * eps)

# Simple example
w, x, b, y = 0.5, 2.0, 0.1, 1.0

# Forward
z = w * x + b
a = max(0, z)  # ReLU
L = (a - y) ** 2

# Analytical gradients (chain rule)
dL_da = 2 * (a - y)
da_dz = 1.0 if z > 0 else 0.0
dz_dw = x
dz_db = 1.0

dL_dw_analytical = dL_da * da_dz * dz_dw
dL_db_analytical = dL_da * da_dz * dz_db

# Numerical verification
params = [w]
def loss_fn_w():
    z = params[0] * x + b
    a = max(0, z)
    return (a - y) ** 2

dL_dw_numerical = numerical_gradient(loss_fn_w, params, 0)

print("Chain Rule Verification:")
print(f"  Forward: z={z:.4f}, a={a:.4f}, L={L:.4f}")
print(f"  dL/dw (analytical): {dL_dw_analytical:.6f}")
print(f"  dL/dw (numerical):  {dL_dw_numerical:.6f}")
print(f"  Match: {np.isclose(dL_dw_analytical, dL_dw_numerical, atol=1e-4)}")
print(f"\n  dL/db (analytical): {dL_db_analytical:.6f}")
print("\n✓ Backprop IS the chain rule. That's all there is to it.")

## Part 2: Full Multi-Layer Network with Backprop

Now we build a **complete neural network class** with proper forward AND backward passes.

This is the code that will let us solve XOR (which a single neuron couldn't).

In [ ]:
# ============================================================
# A complete neural network from scratch
# ============================================================

class Layer:
    """A single fully-connected layer."""
    
    def __init__(self, n_in, n_out, activation='relu'):
        # He initialization for ReLU, Xavier for sigmoid
        if activation == 'relu':
            scale = np.sqrt(2.0 / n_in)  # He init
        else:
            scale = np.sqrt(1.0 / n_in)  # Xavier init
        
        self.W = np.random.randn(n_in, n_out) * scale
        self.b = np.zeros(n_out)
        self.activation = activation
        
        # Cache for backprop
        self.x = None
        self.z = None
        
        # Gradients
        self.dW = None
        self.db = None
    
    def forward(self, x):
        """Forward pass."""
        self.x = x  # cache input
        self.z = x @ self.W + self.b  # linear transform
        
        # Apply activation
        if self.activation == 'relu':
            return np.maximum(0, self.z)
        elif self.activation == 'sigmoid':
            return 1.0 / (1.0 + np.exp(-np.clip(self.z, -500, 500)))
        elif self.activation == 'linear':
            return self.z
    
    def backward(self, da):
        """Backward pass — compute gradients.
        
        da: gradient of loss w.r.t. this layer's output (activation)
        Returns: gradient of loss w.r.t. this layer's INPUT (for previous layer)
        """
        # Step 1: gradient through activation
        if self.activation == 'relu':
            dz = da * (self.z > 0).astype(float)
        elif self.activation == 'sigmoid':
            s = 1.0 / (1.0 + np.exp(-np.clip(self.z, -500, 500)))
            dz = da * s * (1 - s)
        elif self.activation == 'linear':
            dz = da
        
        # Step 2: gradients for parameters
        m = self.x.shape[0]  # batch size
        self.dW = (self.x.T @ dz) / m    # dL/dW
        self.db = np.mean(dz, axis=0)     # dL/db
        
        # Step 3: gradient for input (to pass to previous layer)
        dx = dz @ self.W.T
        return dx


class NeuralNetwork:
    """Multi-layer neural network with backpropagation."""
    
    def __init__(self, layer_sizes, activations=None):
        """
        layer_sizes: e.g., [2, 4, 4, 1] for 2 inputs, two hidden layers of 4, 1 output
        activations: e.g., ['relu', 'relu', 'sigmoid']
        """
        if activations is None:
            activations = ['relu'] * (len(layer_sizes) - 2) + ['sigmoid']
        
        self.layers = []
        for i in range(len(layer_sizes) - 1):
            self.layers.append(Layer(layer_sizes[i], layer_sizes[i+1], activations[i]))
    
    def forward(self, X):
        """Forward pass through all layers."""
        out = X
        for layer in self.layers:
            out = layer.forward(out)
        return out
    
    def backward(self, y_pred, y_true):
        """Backward pass — backpropagate gradients through all layers."""
        # Start with loss gradient (MSE: dL/da = 2(a - y) / N)
        m = y_true.shape[0]
        da = 2 * (y_pred - y_true) / m
        
        # Propagate backward through layers (REVERSE order!)
        for layer in reversed(self.layers):
            da = layer.backward(da)
    
    def update(self, lr):
        """Update all weights using computed gradients."""
        for layer in self.layers:
            layer.W -= lr * layer.dW
            layer.b -= lr * layer.db
    
    def train(self, X, y, lr=0.01, epochs=1000, verbose=True):
        """Full training loop."""
        losses = []
        for epoch in range(epochs):
            # Forward
            y_pred = self.forward(X)
            
            # Loss
            loss = np.mean((y_pred - y.reshape(-1, 1)) ** 2)
            losses.append(loss)
            
            # Backward
            self.backward(y_pred, y.reshape(-1, 1))
            
            # Update
            self.update(lr)
            
            if verbose and epoch % (epochs // 5) == 0:
                print(f"  Epoch {epoch:5d} | Loss: {loss:.6f}")
        
        return losses

# Quick test
net = NeuralNetwork([2, 4, 1], activations=['relu', 'sigmoid'])
print(f"Network architecture: {[l.W.shape for l in net.layers]}")
print("Ready for training!")

## Part 3: Solving XOR — The Multi-Layer Breakthrough

Remember: a single neuron FAILED at XOR because XOR isn't linearly separable.  
With two layers, we can solve it. Let's prove this.

In [ ]:
# ============================================================
# XOR with a multi-layer network
# ============================================================
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([0, 1, 1, 0], dtype=float)

# Architecture: 2 inputs → 8 hidden (ReLU) → 1 output (sigmoid)
net_xor = NeuralNetwork([2, 8, 1], activations=['relu', 'sigmoid'])

print("Training on XOR...")
losses = net_xor.train(X_xor, y_xor, lr=0.5, epochs=3000)

print("\n--- XOR Results (Multi-Layer Network) ---")
for x, y_true in zip(X_xor, y_xor):
    pred = net_xor.forward(x.reshape(1, -1))[0, 0]
    correct = "✓" if (pred > 0.5) == y_true else "✗"
    print(f"  {x} → {pred:.4f} (target: {y_true}) {correct}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(losses)
axes[0].set_title('XOR Training Loss — SOLVED!')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE')
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)

xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 200), np.linspace(-0.5, 1.5, 200))
grid = np.c_[xx.ravel(), yy.ravel()]
zz = np.array([net_xor.forward(p.reshape(1, -1))[0, 0] for p in grid]).reshape(xx.shape)

axes[1].contourf(xx, yy, zz, levels=20, cmap='RdYlBu_r', alpha=0.8)
axes[1].contour(xx, yy, zz, levels=[0.5], colors='black', linewidths=2)
for x, y_true in zip(X_xor, y_xor):
    color = 'red' if y_true == 1 else 'blue'
    axes[1].scatter(x[0], x[1], c=color, s=200, edgecolors='black', zorder=5)
axes[1].set_title('XOR Decision Boundary — Non-Linear!')
axes[1].set_xlabel('x₁')
axes[1].set_ylabel('x₂')

plt.tight_layout()
plt.savefig('w1_02_xor_solved.png', dpi=100, bbox_inches='tight')
plt.show()
print("\n✓ Multiple layers create NON-LINEAR decision boundaries!")

## Part 4: Gradient Checking — Trust but Verify

Never trust your backprop implementation without checking it against numerical gradients.

In [ ]:
# ============================================================
# Gradient checking: compare analytical vs numerical gradients
# ============================================================
def gradient_check(net, X, y, eps=1e-5):
    """Compare backprop gradients with numerical gradients."""
    # First, do a normal forward + backward pass
    y_pred = net.forward(X)
    net.backward(y_pred, y.reshape(-1, 1))
    
    print("Gradient Check (analytical vs numerical):")
    print("-" * 55)
    
    all_close = True
    for layer_idx, layer in enumerate(net.layers):
        # Check a few weights in each layer
        for i in range(min(2, layer.W.shape[0])):
            for j in range(min(2, layer.W.shape[1])):
                # Numerical gradient
                original = layer.W[i, j]
                
                layer.W[i, j] = original + eps
                loss_plus = np.mean((net.forward(X) - y.reshape(-1, 1)) ** 2)
                
                layer.W[i, j] = original - eps
                loss_minus = np.mean((net.forward(X) - y.reshape(-1, 1)) ** 2)
                
                layer.W[i, j] = original  # restore
                
                numerical = (loss_plus - loss_minus) / (2 * eps)
                analytical = layer.dW[i, j]
                
                # Relative error
                denom = max(abs(numerical) + abs(analytical), 1e-8)
                rel_error = abs(numerical - analytical) / denom
                
                ok = "✓" if rel_error < 1e-4 else "✗ MISMATCH"
                if rel_error >= 1e-4:
                    all_close = False
                
                print(f"  Layer {layer_idx} W[{i},{j}]: "
                      f"analytical={analytical:+.6f} "
                      f"numerical={numerical:+.6f} "
                      f"rel_err={rel_error:.2e} {ok}")
    
    print("-" * 55)
    print(f"Overall: {'✓ All gradients match!' if all_close else '✗ Some gradients mismatch'}")
    return all_close

# Test on our XOR network
net_check = NeuralNetwork([2, 4, 1], activations=['relu', 'sigmoid'])
gradient_check(net_check, X_xor, y_xor)

## ✅ Self-Check

- [ ] You can draw the computation graph for forward AND backward pass
- [ ] You can explain backprop as "the chain rule applied backward through the network"
- [ ] Your gradient check passes (analytical ≈ numerical)
- [ ] You solved XOR with a multi-layer network
- [ ] You understand WHY depth helps: each layer adds a non-linear transformation

## ➡️ Next: `W1_03_Pure_Multi_Layer_Network.ipynb` — Real datasets, UAT, and depth experiments

## Visualizing the Neural Network Architecture

A diagram of the network with 1 hidden layer we just built and backpropagated through.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

G = nx.DiGraph()
layers = {'Input': ['x1', 'x2'], 'Hidden': ['h1', 'h2', 'h3'], 'Output': ['y']}
pos = {}
for i, (layer, nodes) in enumerate(layers.items()):
    for j, node in enumerate(nodes):
        pos[node] = (i * 2, len(nodes) / 2.0 - j)
        G.add_node(node)

for in_node in layers['Input']:
    for h_node in layers['Hidden']:
        G.add_edge(in_node, h_node)
for h_node in layers['Hidden']:
    G.add_edge(h_node, layers['Output'][0])

plt.figure(figsize=(8, 4))
nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=1500, arrowsize=15)
plt.title("Multi-Layer Network Architecture (Hidden Layer = 3 nodes)")

plt.savefig('w1_02_network_arch.png', dpi=100, bbox_inches='tight')
plt.show()
